In [1]:
import os, torch
import matplotlib.pyplot as plt
import spintorch
from src.cfg import load_configs
from src.geometry import make_wavegeom
from src.sources import build_sources, temporal_envelope
from src.readouts import build_disk_probes
from src.trainer import train_focus

def assert_baseline_no_weights(model):
    # WaveGeometryMs only carries Ms and B0 (uniform). 
    # If you later add extra field maps, check here they are zero.
    geom = model.geom
    assert hasattr(geom, "Ms") and hasattr(geom, "B0"), "Geometry missing Ms/B0."
    # If your solver stores an additive field map, validate zeros:
    if hasattr(model, "H_add"):
        add_norm = float(torch.as_tensor(model.H_add).abs().max())
        assert add_norm < 1e-12, f"Nonzero additive field ({add_norm}) found."
    print("Baseline regime confirmed: uniform film, no Msat/weight modifiers.")

def main():
    cfg = load_configs(".")
    dev = torch.device(cfg["device"])
    print("Running on", dev)

    # dirs
    base = cfg["io"]["basedir"]
    plotdir = os.path.join("runs","plots", base); os.makedirs(plotdir, exist_ok=True)
    savedir = os.path.join("runs","models", base); os.makedirs(savedir, exist_ok=True)

    # geometry
    geom = make_wavegeom(cfg)

    # sources & probes
    nx, ny = cfg["grid"]["nx"], cfg["grid"]["ny"]
    dx, dy = cfg["grid"]["dx_m"], cfg["grid"]["dy_m"]
    src = build_sources(cfg, nx, ny, dx, dy)
    probes = build_disk_probes(cfg, nx, ny)

    # solver
    dt = cfg["time"]["dt_s"]; timesteps = cfg["time"]["timesteps"]
    model = spintorch.MMSolver(geom, dt, src, probes).to(dev)
    model.retain_history = True

    print("n_params(model) =", sum(p.numel() for p in model.parameters()))
    for name, p in model.named_parameters():
        print(name, p.shape, "requires_grad=", p.requires_grad)

    # temporal drive
    X, t = temporal_envelope(cfg, dt, timesteps, dev)
    # replicate for N sources
    num_src = len(src)
    INPUTS = X.repeat(1, 1, num_src)   # [1, T, num_src]

    # pick target probe index
    outputs_idx = (cfg["probes"]["Ndisk"] // 4)

    # quick debug plots
    t_ns = t.squeeze().cpu().numpy()*1e9
    env = X.squeeze().detach().cpu().numpy()
    plt.figure(); plt.plot(t_ns, env); plt.xlabel("Time (ns)"); plt.ylabel("B_exc (T)")
    plt.title("Gaussian Temporal Excitation"); plt.grid(); plt.savefig(os.path.join(plotdir,"temporal_envelope.png"), dpi=300)

    # train
    train_focus(model, INPUTS, outputs_idx, cfg, savedir, plotdir, nepoch=20, retain_history=True)

if __name__ == "__main__":
    main()


Running on cuda


c:\Users\Tojo\spintorch\source.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('x', torch.tensor(x, dtype=torch.int64))
c:\Users\Tojo\spintorch\source.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('y', torch.tensor(y, dtype=torch.int64))


n_params(model) = 10027
geom.rho torch.Size([100, 100]) requires_grad= True
exch_2D.LAPLACE.weight torch.Size([3, 1, 3, 3]) requires_grad= False


ValueError: optimizer got an empty parameter list

In [7]:
# =========================
# FFT & Magnetostatic Dispersion Analysis (DE/BV)
# =========================
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- utilities ---
def hann(n): return 0.5 - 0.5*np.cos(2*np.pi*np.arange(n)/(n-1))

def P_kt(k, t): 
    kt = np.clip(np.abs(k)*t, 1e-12, None)
    return 1.0 - (1.0 - np.exp(-kt))/kt

def omega_DE(k, H, Ms, gamma, t):
    # Damon–Eshbach (k ⟂ M) - surface spin waves
    mu0 = 4e-7*np.pi
    P = P_kt(k, t)
    return gamma*mu0*np.sqrt((H + Ms*(1 - P))*(H + Ms*P))

def omega_BV(k, H, Ms, gamma, t):
    # Backward Volume (k ∥ M)
    mu0 = 4e-7*np.pi
    P = P_kt(k, t)
    return gamma*mu0*np.sqrt(H*(H + Ms*P))

def extract_line_mz(mz_full, x_index=None, x_span=None):
    """mz_full: [T, nx, ny] or [T, ny] → returns [T, ny] (CPU, float64)"""
    if mz_full.ndim == 3:
        if x_span is not None:
            x0, x1 = x_span
            data = mz_full[:, x0:x1, :].mean(dim=1)
        else:
            xi = int(mz_full.shape[1]//2) if x_index is None else x_index
            data = mz_full[:, xi:xi+1, :].mean(dim=1)
    elif mz_full.ndim == 2:
        data = mz_full
    else:
        raise ValueError("mz_full must be [T, ny] or [T, nx, ny].")
    return data.detach().cpu().double()

# --- pull run data ---
# Use your existing history:
# mz_full = torch.stack(model.m_history, 1)[0,:,2,] - model.m0[0,2,].unsqueeze(0).cpu()
# If you stored full [T, nx, ny] for a component, use that; else adapt as above.
T_total = len(model.m_history)
mz_full = torch.stack(model.m_history, 1)[0, :, 2, ] - model.m0[0, 2, ].unsqueeze(0).cpu()  # [T, nx, ny]
mz_line = extract_line_mz(mz_full, x_span=(mz_full.shape[1]//2 - 2, mz_full.shape[1]//2 + 3))  # [T, ny]

# params
dt = model.dt
dy = model.geom.dy
ny = mz_line.shape[1]

# --- window + zero pad ---
win_t = hann(mz_line.shape[0])[:, None]
win_y = hann(ny)[None, :]
sig = (mz_line.numpy() * (win_t*win_y))

zpad_t, zpad_y = 2, 2
NT = int(2**np.ceil(np.log2(sig.shape[0]*zpad_t)))
NYp= int(2**np.ceil(np.log2(ny*zpad_y)))

# --- FFT: time → freq, then y → ky ---
SIG_fy = np.fft.rfft(sig, n=NT, axis=0)                # [Nf, ny]
freqs = np.fft.rfftfreq(NT, d=dt)                      # Hz
SIG_fk = np.fft.fftshift(np.fft.fft(SIG_fy, n=NYp, axis=1), axes=1)
ky = np.fft.fftshift(np.fft.fftfreq(NYp, d=dy)) * 2*np.pi  # rad/m
PSD = np.abs(SIG_fk)**2

# axis in human units
freq_GHz = freqs/1e9
ky_per_um = ky/1e6

# --- theory overlay ---
Ms   = float(model.geom.Ms)      # A/m
tfilm= float(model.geom.dz)      # m (adjust to your physical thickness if needed)
B0   = float(model.geom.B0)      # T
mu0  = 4e-7*np.pi
H0   = B0/mu0                    # A/m
gamma= 1.760859e11               # rad/(s·T)

geometry_mode = "DE"  # or "BV"
if geometry_mode == "DE":
    f_theory = omega_DE(ky, H0, Ms, gamma, tfilm)/(2*np.pi)
else:
    f_theory = omega_BV(ky, H0, Ms, gamma, tfilm)/(2*np.pi)
f_theory_GHz = f_theory/1e9

# --- plot k–f with overlay ---
plt.figure(figsize=(7.6,5.8))
extent=[ky_per_um.min(), ky_per_um.max(), freq_GHz.min(), freq_GHz.max()]
plt.imshow(10*np.log10(PSD.T+1e-20), origin='lower', aspect='auto', extent=extent)
plt.plot(ky_per_um, f_theory_GHz, lw=2)
plt.xlabel(r"$k_y$ (rad / $\mu$m)"); plt.ylabel("Frequency (GHz)")
plt.title(f"k–f map + {geometry_mode} theory")
plt.colorbar(label="Power (dB)")
plt.tight_layout(); plt.show()

# --- ridge vs theory error ---
power_per_k = PSD.sum(axis=0)
valid = power_per_k >= np.percentile(power_per_k, 75)
peak_idx = PSD[:, valid].argmax(axis=0)
f_peak = freq_GHz[peak_idx]
k_sel = ky[valid]
if geometry_mode == "DE":
    f_sel_th = omega_DE(k_sel, H0, Ms, gamma, tfilm)/(2*np.pi)/1e9
else:
    f_sel_th = omega_BV(k_sel, H0, Ms, gamma, tfilm)/(2*np.pi)/1e9

rel_err = np.abs(f_peak - f_sel_th)/np.maximum(f_sel_th, 1e-12)
print(f"mean rel. error = {100*rel_err.mean():.2f}%   (pass if < 10%)")
print(f"max  rel. error = {100*rel_err.max():.2f}%   on {valid.sum()} k-columns")


AttributeError: 'WaveGeometryMs' object has no attribute 'dy'

In [1]:
"""Optimize a focusing model"""
import torch
import os
import spintorch
import numpy as np
from spintorch.utils import tic, toc, stat_cuda
from spintorch.plot import wave_integrated, wave_snapshot

import warnings
warnings.filterwarnings("ignore", message=".*Casting complex values to real.*")


"""Parameters"""
dx = 50e-9      # discretization (m)
dy = 50e-9      # discretization (m)
dz = 20e-9      # discretization (m)
nx = 100        # size x    (cells)
ny = 100        # size y    (cells)

Ms = 140e3      # saturation magnetization (A/m)
B0 = 60e-3      # bias field (T)
Bt = 1e-3       # excitation field amplitude (T)

dt = 20e-12     # timestep (s)
f1 = 4e9        # source frequency (Hz)
timesteps = 600 # number of timesteps for wave propagation


'''Directories'''
basedir = 'focus_Ms/'
plotdir = 'plots/' + basedir
if not os.path.isdir(plotdir):
    os.makedirs(plotdir)
savedir = 'models/' + basedir
if not os.path.isdir(savedir):
    os.makedirs(savedir)    

'''Geometry, sources, probes, model definitions'''
### Here are three geometry modules initialized, just uncomment one of them to try:
# Ms_CoPt = 723e3 # saturation magnetization of the nanomagnets (A/m)
# r0, dr, dm, z_off = 15, 4, 2, 10  # starting pos, period, magnet size, z distance
# rx, ry = int((nx-2*r0)/dr), int((ny-2*r0)/dr+1)
# rho = torch.zeros((rx, ry))  # Design parameter array
# geom = spintorch.WaveGeometryArray(rho, (nx, ny), (dx, dy, dz), Ms, B0, 
#                                     r0, dr, dm, z_off, rx, ry, Ms_CoPt)
# B1 = 50e-3      # training field multiplier (T)
# geom = spintorch.WaveGeometryFreeForm((nx, ny), (dx, dy, dz), B0, B1, Ms)
geom = spintorch.WaveGeometryMs((nx, ny), (dx, dy, dz), Ms, B0)
src = spintorch.WaveLineSource(10, 0, 10, ny-1, dim=2)
probes = []
Np = 19  # number of probes
for p in range(Np):
    probes.append(spintorch.WaveIntensityProbeDisk(nx-15, int(ny*(p+1)/(Np+1)), 2))
model = spintorch.MMSolver(geom, dt, [src], probes)

dev = torch.device('cuda')  # 'cuda' or 'cpu'
print('Running on', dev)
model.to(dev)   # sending model to GPU/CPU

Running on cuda


MMSolver(
  (geom): WaveGeometryMs()
  (sources): ModuleList(
    (0): WaveLineSource()
  )
  (probes): ModuleList(
    (0-18): 19 x WaveIntensityProbeDisk()
  )
  (demag_2D): Demag()
  (exch_2D): Exchange(
    (LAPLACE): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False, padding_mode=replicate)
  )
  (Alpha): Damping()
  (torque_SOT): SOT()
)

In [6]:
print("n_params(model) =", sum(p.numel() for p in model.parameters()))
for name, p in model.named_parameters():
    print(name, p.shape, "requires_grad=", p.requires_grad)


n_params(model) = 10027
geom.rho torch.Size([100, 100]) requires_grad= True
exch_2D.LAPLACE.weight torch.Size([3, 1, 3, 3]) requires_grad= False
